In [ ]:
import torch
import pandas as pd
import os
from PIL import Image
from transformers import LlavaProcessor, LlavaForConditionalGeneration
from tqdm import tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip "/content/drive/MyDrive/place-pulse-2.0 (2).zip" -d /content/

Streaming output truncated to the last 5000 lines.
  inflating: /content/__MACOSX/images/._35.749498_139.709852_513e6c3efdc9f0358700c139_Tokyo.JPG  
  inflating: /content/images/44.423834_26.025045_5140ceb7fdc9f0492600308a_Bucharest.JPG  
  inflating: /content/__MACOSX/images/._44.423834_26.025045_5140ceb7fdc9f0492600308a_Bucharest.JPG  
  inflating: /content/images/52.218359_20.909351_50f42b52fdc9f065f0001369_Warsaw.JPG  
  inflating: /content/__MACOSX/images/._52.218359_20.909351_50f42b52fdc9f065f0001369_Warsaw.JPG  
  inflating: /content/images/55.799112_37.779071_513e1dc9fdc9f03587009aba_Moscow.JPG  
  inflating: /content/__MACOSX/images/._55.799112_37.779071_513e1dc9fdc9f03587009aba_Moscow.JPG  
  inflating: /content/images/41.955114_-87.758089_513cb5dbfdc9f035870009a6_Chicago.JPG  
  inflating: /content/__MACOSX/images/._41.955114_-87.758089_513cb5dbfdc9f035870009a6_Chicago.JPG  
  inflating: /content/images/39.757205_-105.109594_513d6b71fdc9f03587004d07_Denver.JPG  
  inflating:

In [ ]:
# Load votes
df_votes = pd.read_csv("/content/votes.tsv", sep="\t")

# Load studies
df_studies = pd.read_csv("/content/studies.tsv", sep="\t")

# Image directory
image_dir = "/content/images"

In [ ]:
print(df_votes.shape)
print(df_studies.shape)
print(len(os.listdir(image_dir)))

(1555561, 7)
(6, 6)
110988


In [ ]:
df_votes["choice"].value_counts()

,count
choice,
right,686368
left,664178
equal,204992
rigPr,5
equPr,5
lePr,4
response.write(9854781*9182327),1
set|set&set,1
'+response.write(9854781*9182327)+',1


In [ ]:
df_votes = df_votes[df_votes["choice"].isin(["left", "right"])]
print(df_votes["choice"].value_counts())

choice
right    686368
left     664178
Name: count, dtype: int64


In [ ]:
# ==========================================
# Build Pair-Level Dataset
# ==========================================

# Create pair identifier
df_votes["pair_id"] = df_votes.apply(
    lambda x: "_".join(
        sorted([str(x["left"]), str(x["right"])])
    ) + "_" + str(x["study_id"]),
    axis=1
)

# Count left wins
left_votes = (
    df_votes[df_votes["choice"].isin(["left", "A", 0])]
    .groupby(["pair_id", "left", "right", "study_id"])
    .size()
    .reset_index(name="left_votes")
)

# Count right wins
right_votes = (
    df_votes[df_votes["choice"].isin(["right", "B", 1])]
    .groupby(["pair_id", "left", "right", "study_id"])
    .size()
    .reset_index(name="right_votes")
)

# Merge vote counts
pair_df = pd.merge(
    left_votes,
    right_votes,
    on=["pair_id", "left", "right", "study_id"],
    how="outer"
)

pair_df["left_votes"] = pair_df["left_votes"].fillna(0)
pair_df["right_votes"] = pair_df["right_votes"].fillna(0)

print("Pairs before filtering:", len(pair_df))
pair_df.head()

Pairs before filtering: 1333630


,pair_id,left,right,study_id,left_votes,right_votes
0,${9999640+9999388}_5140d960fdc9f04926003bb4_50...,${9999640+9999388},5140d960fdc9f04926003bb4,50f62c41a84ea7c5fdd2e454,1.0,0.0
1,50e5f7d4d7c3df413b00056a_50f42c0dfdc9f065f0001...,50e5f7d4d7c3df413b00056a,50f42c0dfdc9f065f00017bc,50a68a51fdc9f05596000002,1.0,0.0
2,50e5f7d4d7c3df413b00056a_50f42c6afdc9f065f0001...,50f42c6afdc9f065f0001bd4,50e5f7d4d7c3df413b00056a,50a68a51fdc9f05596000002,1.0,0.0
3,50e5f7d4d7c3df413b00056a_50f55e9afdc9f065f0004...,50f55e9afdc9f065f0004d7b,50e5f7d4d7c3df413b00056a,50f62c68a84ea7c5fdd2e456,1.0,0.0
4,50e5f7d4d7c3df413b00056a_50f562ddfdc9f065f0005...,50e5f7d4d7c3df413b00056a,50f562ddfdc9f065f0005af3,50a68a51fdc9f05596000002,0.0,1.0


In [ ]:
pair_df["total_votes"] = pair_df["left_votes"] + pair_df["right_votes"]

pair_df["winner"] = pair_df.apply(
    lambda x: "left" if x["left_votes"] > x["right_votes"]
    else ("right" if x["right_votes"] > x["left_votes"] else "tie"),
    axis=1
)

pair_df["winner"].value_counts()

,count
winner,
right,677407
left,655829
tie,394


In [ ]:
pair_df = pair_df[pair_df["winner"] != "tie"]

print("Pairs after removing ties:", len(pair_df))

Pairs after removing ties: 1333236


In [ ]:
pair_df["majority_ratio"] = pair_df.apply(
    lambda x: max(x["left_votes"], x["right_votes"]) / x["total_votes"],
    axis=1
)

pair_df = pair_df[pair_df["majority_ratio"] >= 0.7].copy()

print("Final clean pairs:", len(pair_df))

Final clean pairs: 1332944


In [ ]:
print("Total pairs:", len(pair_df))

print("Average votes per pair:",
      pair_df["total_votes"].mean())

print("Tie percentage:",
      (pair_df["winner"] == "tie").mean())

Total pairs: 1332944
Average votes per pair: 1.0112675401217155
Tie percentage: 0.0


In [ ]:
df_votes["left_norm"] = df_votes[["left","right"]].min(axis=1)
df_votes["right_norm"] = df_votes[["left","right"]].max(axis=1)

In [ ]:
pair_counts = df_votes.groupby(["left_norm", "right_norm", "study_id"]).size()

print("Pairs with more than one vote:",
      (pair_counts > 1).sum())

print("Max votes for a single pair:",
      pair_counts.max())

Pairs with more than one vote: 5618
Max votes for a single pair: 69


In [ ]:
pair_votes = (
    df_votes.groupby(["left_norm", "right_norm", "study_id", "choice"])
    .size()
    .unstack(fill_value=0)
)

print("Vote columns:", pair_votes.columns)

# Get the two vote columns dynamically
col1, col2 = pair_votes.columns[:2]

pair_votes["tie"] = pair_votes[col1] == pair_votes[col2]

print("True tie pairs:", pair_votes["tie"].sum())

Vote columns: Index(['left', 'right'], dtype='object', name='choice')
True tie pairs: 405


In [ ]:
image_map = {}

for file in os.listdir(image_dir):
    if file.lower().endswith(".jpg"):
        parts = file.split("_")
        if len(parts) >= 3:
            image_id = parts[2]  # <-- THIS is the correct ID
            image_map[image_id] = os.path.join(image_dir, file)

print("Total images mapped:", len(image_map))

Total images mapped: 110988


In [ ]:
print(len(image_map))

110988


In [ ]:
print("Total cleaned votes:", len(df_votes))

print("Unique pairs:",
      df_votes.groupby(["left","right","study_id"]).ngroups)

print("Average votes per pair:",
      df_votes.groupby(["left","right","study_id"]).size().mean())

Total cleaned votes: 1350546
Unique pairs: 1333630
Average votes per pair: 1.012684177770446


In [ ]:
model_id = "llava-hf/llava-1.5-7b-hf"

processor = LlavaProcessor.from_pretrained(model_id)

model = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
).to("cuda")

print("Model loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/674 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

Model loaded.


In [ ]:
def concatenate_images(path_A, path_B):
    img_A = Image.open(path_A).convert("RGB")
    img_B = Image.open(path_B).convert("RGB")

    new_img = Image.new("RGB", (img_A.width + img_B.width, img_A.height))
    new_img.paste(img_A, (0, 0))
    new_img.paste(img_B, (img_A.width, 0))

    return new_img

In [ ]:
ATTRIBUTE_CUES = {
    "beautiful": [
        "architecture",
        "greenery",
        "cleanliness",
        "visual harmony",
        "scenic features"
    ],
    "safer": [
    "well-lit areas",
    "visible people nearby",
    "clean and maintained streets",
    "no signs of damage or neglect",
    "active businesses or shops"
],
    "wealthier": [
        "modern buildings",
        "expensive cars",
        "clean streets",
        "landscaping",
        "well maintained infrastructure"
    ],
    "livelier": [
       "many people visible",
        "movement or activity",
        "vehicles or pedestrians",
        "open shops or markets",
        "bright lights or signs"
    ],
    "more boring": [
        "plain buildings",
        "empty streets",
        "lack of activity",
        "few people",
        "dull environment"
    ],
    "more depressing": [
        "abandoned buildings",
        "poor lighting",
        "empty streets",
        "graffiti",
        "neglected infrastructure"
    ]
}


In [ ]:
attribute_map = {
    "beautiful": "5217c351ad93a7d3e7b07a64",
    "safer": "50a68a51fdc9f05596000002",
    "livelier": "50f62c41a84ea7c5fdd2e454",
    "more boring": "50f62c68a84ea7c5fdd2e456",
    "wealthier": "50f62cb7a84ea7c5fdd2e458",
    "more depressing": "50f62ccfa84ea7c5fdd2e459",

}

In [ ]:
def get_prediction(image1, image2, attribute):

    prompt = f"""
    Image A and Image B are shown.

    Compare the two images carefully.

    Which image looks more {attribute}?

    Answer with only A or B.
    """

    combined = concatenate_images(image1, image2)

    inputs = processor(
        images=combined,
        text=prompt,
        return_tensors="pt"
    ).to(model.device)

    output = model.generate(**inputs, max_new_tokens=10)

    answer = processor.decode(output[0], skip_special_tokens=True)

    return answer

In [ ]:
def evaluate_attribute(attribute_name, study_id, n_samples=50):

    votes = df_votes[df_votes["study_id"] == study_id].copy()
    print(f"{attribute_name} — votes after study filter:", len(votes))

    votes = votes[
        votes["left"].isin(image_map.keys()) &
        votes["right"].isin(image_map.keys())
    ].copy()

    print(f"{attribute_name} — votes after image filter:", len(votes))

    records = []

    for _, row in votes.iterrows():
        img_A = image_map[row["left"]]
        img_B = image_map[row["right"]]
        label = 1 if row["choice"] == "left" else 0

        records.append({
            "img_A": img_A,
            "img_B": img_B,
            "label": label
        })

    votes_df = pd.DataFrame(records)

    if len(votes_df) < n_samples:
        n_samples = len(votes_df)

    eval_df = votes_df.sample(n=n_samples, random_state=42).reset_index(drop=True)

    print("votes_df length:", len(votes_df))
    print("eval_df length:", len(eval_df))

    ########################################
    # ✅ FINAL PARSER (SUPPORTS BOTH MODES)
    ########################################

    def parse_prediction(text):
        if text is None:
            return None

        text = text.lower().strip()

        if "assistant:" in text:
            text = text.split("assistant:")[-1].strip()

        # CoT format
        if "answer: a" in text:
            return 1
        if "answer: b" in text:
            return 0

        # Direct format
        if "image a" in text:
            return 1
        if "image b" in text:
            return 0

        # Fallback
        if text == "a" or text.startswith("a"):
            return 1
        if text == "b" or text.startswith("b"):
            return 0

        return None

    ########################################
    # DIRECT
    ########################################

    direct_results = []

    for i in tqdm(range(len(eval_df)), desc=f"Direct: {attribute_name}"):

        img_A = eval_df.iloc[i]["img_A"]
        img_B = eval_df.iloc[i]["img_B"]
        gt = eval_df.iloc[i]["label"]

        prompt = f"""USER: <image>
Which place looks {attribute_name}?

Answer ONLY with:
Image A
or
Image B
ASSISTANT:"""

        combined_img = concatenate_images(img_A, img_B)

        inputs = processor(text=prompt, images=combined_img, return_tensors="pt").to("cuda", torch.float16)
        output = model.generate(**inputs, max_new_tokens=5, temperature=0.0, do_sample=False)
        pred1 = parse_prediction(processor.decode(output[0], skip_special_tokens=True))

        combined_img_swapped = concatenate_images(img_B, img_A)

        inputs = processor(text=prompt, images=combined_img_swapped, return_tensors="pt").to("cuda", torch.float16)
        output = model.generate(**inputs, max_new_tokens=5, temperature=0.0, do_sample=False)
        pred2 = parse_prediction(processor.decode(output[0], skip_special_tokens=True))

        if pred2 is not None:
            pred2 = 1 - pred2

        is_tie = (pred1 is not None and pred2 is not None and pred1 != pred2)

        votes = [v for v in [pred1, pred2] if v is not None]

        if len(votes) == 0:
            final_pred = None
        elif votes.count(1) > votes.count(0):
            final_pred = 1
        elif votes.count(0) > votes.count(1):
            final_pred = 0
        else:
            final_pred = pred1

        direct_results.append({"pred": final_pred, "gt": gt, "tie": is_tie})

    ########################################
    # CoT (FIXED)
    ########################################

    cot_results = []

    for i in tqdm(range(len(eval_df)), desc=f"CoT: {attribute_name}"):

        img_A = eval_df.iloc[i]["img_A"]
        img_B = eval_df.iloc[i]["img_B"]
        gt = eval_df.iloc[i]["label"]

        cues = ATTRIBUTE_CUES.get(attribute_name.lower(), [])
        cue_text = ", ".join(cues)

        reasoning_prompt = f"""USER: <image>

Two places are shown side-by-side:

Image A (left)
Image B (right)

Your task is to determine which place looks more **{attribute_name}**.

Important visual cues related to "{attribute_name}" may include:
{cue_text}

Follow these steps:

Step 1: Briefly describe Image A focusing on relevant cues.
Step 2: Briefly describe Image B focusing on relevant cues.
Step 3: Compare both images based on those cues.
Step 4: Decide which image better represents "{attribute_name}".

IMPORTANT:
Give your final answer EXACTLY in this format on a new line:
Answer: A
or
Answer: B

ASSISTANT:
"""

        combined_img = concatenate_images(img_A, img_B)

        inputs = processor(text=reasoning_prompt, images=combined_img, return_tensors="pt").to("cuda", torch.float16)

        output = model.generate(
            **inputs,
            max_new_tokens=80,
            temperature=0.0,
            do_sample=False
        )

        pred1 = parse_prediction(processor.decode(output[0], skip_special_tokens=True))

        # SWAPPED
        combined_img_swapped = concatenate_images(img_B, img_A)

        inputs = processor(text=reasoning_prompt, images=combined_img_swapped, return_tensors="pt").to("cuda", torch.float16)

        output = model.generate(
            **inputs,
            max_new_tokens=80,
            temperature=0.0,
            do_sample=False
        )

        pred2 = parse_prediction(processor.decode(output[0], skip_special_tokens=True))

        if pred2 is not None:
            pred2 = 1 - pred2

        is_tie = (pred1 is not None and pred2 is not None and pred1 != pred2)

        votes = [v for v in [pred1, pred2] if v is not None]

        if len(votes) == 0:
            final_pred = None
        elif votes.count(1) > votes.count(0):
            final_pred = 1
        elif votes.count(0) > votes.count(1):
            final_pred = 0
        else:
            final_pred = pred1

        cot_results.append({"pred": final_pred, "gt": gt, "tie": is_tie})

    ########################################
    # METRICS
    ########################################

    def compute_metrics(results):
        total = sum(r["pred"] is not None for r in results)
        correct = sum(r["pred"] == r["gt"] for r in results if r["pred"] is not None)

        acc = correct / total if total > 0 else 0
        tie_rate = sum(r["tie"] for r in results) / len(results)

        return acc, tie_rate

    direct_acc, direct_tie = compute_metrics(direct_results)
    cot_acc, cot_tie = compute_metrics(cot_results)

    ########################################
    # DEBUG
    ########################################

    print("Direct valid predictions:", sum(r["pred"] is not None for r in direct_results))
    print("CoT valid predictions:", sum(r["pred"] is not None for r in cot_results))
    print("Total samples:", len(eval_df))

    ########################################
    # RETURN
    ########################################

    return {
        "attribute": attribute_name,
        "direct_acc": direct_acc,
        "cot_acc": cot_acc,
        "direct_tie_rate": direct_tie,
        "cot_tie_rate": cot_tie
    }

In [ ]:
attributes = [
    "beautiful",
    "safer",
    "livelier",
    "more boring",
    "wealthier",
    "more depressing",

]

In [ ]:
evaluate_attribute("livelier", attribute_map["livelier"], n_samples=100)

livelier — votes after study filter: 320803
livelier — votes after image filter: 318958
votes_df length: 318958
eval_df length: 100


CoT: livelier: 100%|██████████| 100/100 [00:42<00:00,  2.34it/s]

Direct valid predictions: 100
CoT valid predictions: 100
Total samples: 100


{'attribute': 'livelier',
 'direct_acc': np.float64(0.45),
 'cot_acc': np.float64(0.59),
 'direct_tie_rate': 0.87,
 'cot_tie_rate': 0.69}

In [ ]:
evaluate_attribute("more depressing", attribute_map["more depressing"], n_samples=100)

more depressing — votes after study filter: 125030
more depressing — votes after image filter: 124257
votes_df length: 124257
eval_df length: 100


CoT: more depressing: 100%|██████████| 100/100 [00:44<00:00,  2.22it/s]

Direct valid predictions: 100
CoT valid predictions: 100
Total samples: 100


{'attribute': 'more depressing',
 'direct_acc': np.float64(0.54),
 'cot_acc': np.float64(0.5),
 'direct_tie_rate': 0.37,
 'cot_tie_rate': 0.4}

In [ ]:
results = []

for attr in attributes:

    print("\n==============================")
    print(f"Running evaluation for: {attr}")
    print("==============================\n")

    res = evaluate_attribute(attr, attribute_map[attr], n_samples=100)


    results.append(res)


Running evaluation for: beautiful

beautiful — votes after study filter: 190606
beautiful — votes after image filter: 189539


KeyboardInterrupt: 

In [ ]:
results_df = pd.DataFrame(results)
print(results_df)

         attribute  direct_acc  cot_acc  direct_tie_rate  cot_tie_rate
0        beautiful        0.65     0.59             0.39          0.51
1            safer        0.56     0.54             0.77          0.42
2         livelier        0.45     0.59             0.87          0.69
3      more boring        0.51     0.51             0.98          0.61
4        wealthier        0.54     0.54             0.55          0.46
5  more depressing        0.54     0.50             0.37          0.40


In [ ]:
print("Sample vote left ID:", df_votes["left"].iloc[0])
print("Sample image_map key:", list(image_map.keys())[0])

Sample vote left ID: 513d2dbefdc9f03587002515
Sample image_map key: 5141471efdc9f049260069a4


In [ ]:
# ==========================================
# Independent Scoring Evaluation
# ==========================================
def get_score(image_path, attribute_name):

    image = Image.open(image_path).convert("RGB")

    prompt = f"""USER: <image>
On a scale from 0 to 10,
how {attribute_name.rstrip('r')} does this place look?

Answer ONLY with a number between 0 and 10.
ASSISTANT:"""

    inputs = processor(
        text=prompt,
        images=image,
        return_tensors="pt"
    ).to("cuda", torch.float16)

    output = model.generate(
        **inputs,
        max_new_tokens=10,
        temperature=0.0,
        do_sample=False
    )

    full_text = processor.decode(output[0], skip_special_tokens=True)

# Only keep text after ASSISTANT:
    if "ASSISTANT:" in full_text:
      text = full_text.split("ASSISTANT:")[-1].strip()
    else:
      text = full_text.strip()
    print("RAW OUTPUT:", text)  # ← add this

    import re

    numbers = re.findall(r"\d+", text)

    if numbers:
      return int(numbers[0])   # now first number is correct
    else:
      return 0

In [ ]:
def evaluate_attribute_independent(attribute_name, study_id, n_samples=100):

    ATTRIBUTE_PHRASES = {
        "beautiful":       "beautiful",
        "safer":           "safe",
        "wealthier":       "wealthy",
        "livelier":        "lively",
        "more boring":     "boring",
        "more depressing": "depressing"
    }

    votes = df_votes[df_votes["study_id"] == study_id].copy()
    print(f"{attribute_name} — votes after study filter:", len(votes))

    votes = votes[
        votes["left"].isin(image_map.keys()) &
        votes["right"].isin(image_map.keys())
    ].copy()

    print(f"{attribute_name} — votes after image filter:", len(votes))

    records = []
    for _, row in votes.iterrows():
        img_A = image_map[row["left"]]
        img_B = image_map[row["right"]]
        label = 1 if row["choice"] == "left" else 0
        records.append({"img_A": img_A, "img_B": img_B, "label": label})

    votes_df = pd.DataFrame(records)

    if len(votes_df) < n_samples:
        n_samples = len(votes_df)

    eval_df = votes_df.sample(n=n_samples, random_state=42).reset_index(drop=True)

    print("votes_df length:", len(votes_df))
    print("eval_df length:", len(eval_df))

    def parse_score(text):
        text = text.lower().strip()
        if "assistant:" in text:
            text = text.split("assistant:")[-1].strip()
        import re
        match = re.search(r"\d+", text)
        return int(match.group()) if match else None

    results = []

    for i in tqdm(range(len(eval_df)), desc=f"Independent: {attribute_name}"):

        img_A = eval_df.iloc[i]["img_A"]
        img_B = eval_df.iloc[i]["img_B"]
        gt    = eval_df.iloc[i]["label"]

        cues        = ATTRIBUTE_CUES.get(attribute_name.lower(), [])
        cue_text    = ", ".join(cues)
        attr_phrase = ATTRIBUTE_PHRASES.get(attribute_name.lower(), attribute_name)

        # Original prompt — best results so far
        prompt = f"""USER: <image>

Rate how {attr_phrase} this place looks on a scale from 0 to 100.

Important visual cues may include:
{cue_text}

Be precise and use the full range when appropriate.
Avoid default or middle values unless necessary.
Use a wide range of scores to clearly distinguish between different scenes.
Do not give identical scores unless the two places truly look the same.

Answer ONLY with a number between 0 and 100.

ASSISTANT:"""

        scores = []
        for img_path in [img_A, img_B]:
            image  = Image.open(img_path).convert("RGB")
            inputs = processor(
                text=prompt,
                images=image,
                return_tensors="pt"
            ).to("cuda", torch.float16)

            output = model.generate(
                **inputs,
                max_new_tokens=10,
                temperature=0.0,
                do_sample=False
            )

            score = parse_score(
                processor.decode(output[0], skip_special_tokens=True)
            )
            scores.append(score)

        score_A, score_B = scores

        # Original threshold
        threshold = 3
        if attribute_name == "livelier":
            threshold = 1
        else:
            threshold = 3

        if score_A is None or score_B is None:
            final_pred, is_tie = None, False
        elif abs(score_A - score_B) <= threshold:
            final_pred, is_tie = None, True
        elif score_A > score_B:
            final_pred, is_tie = 1, False
        else:
            final_pred, is_tie = 0, False

        results.append({
            "pred":    final_pred,
            "gt":      gt,
            "tie":     is_tie,
            "score_A": score_A,
            "score_B": score_B
        })

    def compute_metrics(results):
        total    = sum(r["pred"] is not None for r in results)
        correct  = sum(r["pred"] == r["gt"] for r in results if r["pred"] is not None)
        acc      = correct / total if total > 0 else None
        pred_A   = sum(r["pred"] == 1 for r in results)
        pred_B   = sum(r["pred"] == 0 for r in results)
        tie_rate = sum(r["tie"] for r in results) / len(results) if results else 0
        return acc, pred_A, pred_B, tie_rate

    acc, pred_A, pred_B, tie_rate = compute_metrics(results)

    scores_A = [r["score_A"] for r in results if r["score_A"] is not None]
    scores_B = [r["score_B"] for r in results if r["score_B"] is not None]
    if scores_A and scores_B:
        print(f"Score A — min: {min(scores_A)}, max: {max(scores_A)}, mean: {sum(scores_A)/len(scores_A):.1f}")
        print(f"Score B — min: {min(scores_B)}, max: {max(scores_B)}, mean: {sum(scores_B)/len(scores_B):.1f}")

    return {
        "attribute":            attribute_name,
        "independent_acc":      round(acc, 3) if acc else None,
        "independent_A":        pred_A,
        "independent_B":        pred_B,
        "independent_tie_rate": round(tie_rate, 3)
    }

In [ ]:
evaluate_attribute_independent("beautiful", attribute_map["beautiful"], n_samples=100)

beautiful — votes after study filter: 190606
beautiful — votes after image filter: 189539
votes_df length: 189539
eval_df length: 100


Independent: beautiful: 100%|██████████| 100/100 [00:41<00:00,  2.44it/s]

Score A — min: 50, max: 90, mean: 79.3
Score B — min: 50, max: 90, mean: 81.0


{'attribute': 'beautiful',
 'independent_acc': np.float64(0.83),
 'independent_A': 22,
 'independent_B': 31,
 'independent_tie_rate': 0.47}

In [ ]:
results = []

for attr in attributes:

    print("\n==============================")
    print(f"Running evaluation for: {attr}")
    print("==============================\n")

    res = evaluate_attribute_independent(attr, attribute_map[attr], n_samples=100)

    results.append(res)



results_df = pd.DataFrame(results)

results_df = results_df[[
    "attribute",
    "independent_acc",
    "independent_tie_rate"
]].round(3)

results_df


Running evaluation for: beautiful

beautiful — votes after study filter: 190606
beautiful — votes after image filter: 189539
votes_df length: 189539
eval_df length: 100


Independent: beautiful: 100%|██████████| 100/100 [00:41<00:00,  2.40it/s]


Score A — min: 50, max: 90, mean: 79.3
Score B — min: 50, max: 90, mean: 81.0

Running evaluation for: safer

safer — votes after study filter: 443052
safer — votes after image filter: 440758
votes_df length: 440758
eval_df length: 100


Independent: safer: 100%|██████████| 100/100 [00:41<00:00,  2.42it/s]


Score A — min: 50, max: 100, mean: 76.7
Score B — min: 50, max: 100, mean: 77.0

Running evaluation for: livelier

livelier — votes after study filter: 320803
livelier — votes after image filter: 318958
votes_df length: 318958
eval_df length: 100


...

Independent: wealthier: 100%|██████████| 100/100 [00:41<00:00,  2.40it/s]


Score A — min: 50, max: 100, mean: 64.2
Score B — min: 50, max: 100, mean: 63.2

Running evaluation for: more depressing

more depressing — votes after study filter: 125030
more depressing — votes after image filter: 124257
votes_df length: 124257
eval_df length: 100


Independent: more depressing: 100%|██████████| 100/100 [00:40<00:00,  2.46it/s]

Score A — min: 50, max: 80, mean: 58.9
Score B — min: 50, max: 90, mean: 58.9


,attribute,independent_acc,independent_tie_rate
0,beautiful,0.830,0.47
1,safer,0.700,0.50
2,livelier,0.714,0.93
3,more boring,0.644,0.41
4,wealthier,0.725,0.31
5,more depressing,0.596,0.48


In [ ]:
# Debug livelier with ASSISTANT: ending
attr_phrase = "lively"
cues        = ATTRIBUTE_CUES.get("livelier", [])
cue_text    = ", ".join(cues)

prompt = f"""USER: <image>

Rate how {attr_phrase} this place looks on a scale from 0 to 100.

Important visual cues may include:
{cue_text}

Be precise and use the full range when appropriate.
Avoid default or middle values unless necessary.
Use a wide range of scores to clearly distinguish between different scenes.
Do not give identical scores unless the two places truly look the same.

Answer ONLY with a number between 0 and 100.

ASSISTANT:"""

votes  = df_votes[df_votes["study_id"] == attribute_map["livelier"]].copy()
sample = votes[
    votes["left"].isin(image_map.keys()) &
    votes["right"].isin(image_map.keys())
].sample(5, random_state=42)

for _, row in sample.iterrows():
    for img_id, label in [(row["left"], "A"), (row["right"], "B")]:
        image  = Image.open(image_map[img_id]).convert("RGB")
        inputs = processor(
            text=prompt, images=image, return_tensors="pt"
        ).to("cuda", torch.float16)
        output = model.generate(
            **inputs, max_new_tokens=10, temperature=0.0, do_sample=False
        )
        raw   = processor.decode(output[0], skip_special_tokens=True)
        after = raw.split("ASSISTANT:")[-1].strip()
        print(f"Image {label} | Score: {after}")
    print("---")

Image A | Score: 50
Image B | Score: 50
---
Image A | Score: 50
Image B | Score: 50
---
Image A | Score: 50
Image B | Score: 50
---
Image A | Score: 50
Image B | Score: 50
---
Image A | Score: 50
Image B | Score: 50
---


In [ ]:
def evaluate_livelier_independent(study_id, n_samples=100):

    votes = df_votes[df_votes["study_id"] == study_id].copy()
    print(f"livelier — votes after study filter:", len(votes))

    votes = votes[
        votes["left"].isin(image_map.keys()) &
        votes["right"].isin(image_map.keys())
    ].copy()

    print(f"livelier — votes after image filter:", len(votes))

    records = []
    for _, row in votes.iterrows():
        img_A = image_map[row["left"]]
        img_B = image_map[row["right"]]
        label = 1 if row["choice"] == "left" else 0
        records.append({"img_A": img_A, "img_B": img_B, "label": label})

    votes_df = pd.DataFrame(records)
    eval_df  = votes_df.sample(n=min(n_samples, len(votes_df)), random_state=42).reset_index(drop=True)

    print("eval_df length:", len(eval_df))

    def parse_score(text):
        text = text.lower().strip()
        if "assistant:" in text:
            text = text.split("assistant:")[-1].strip()
        import re
        match = re.search(r"\d+", text)
        return int(match.group()) if match else None

    cues     = ATTRIBUTE_CUES.get("livelier", [])
    cue_text = ", ".join(cues)

    # Same prompt but with ASSISTANT: ending instead of A:
    prompt = f"""USER: <image>

Rate how lively this place looks on a scale from 0 to 100.

Important visual cues may include:
{cue_text}

Be precise and use the full range when appropriate.
Avoid default or middle values unless necessary.
Use a wide range of scores to clearly distinguish between different scenes.
Do not give identical scores unless the two places truly look the same.

Answer ONLY with a number between 0 and 100.

ASSISTANT:"""

    results = []

    for i in tqdm(range(len(eval_df)), desc="Independent: livelier"):

        img_A = eval_df.iloc[i]["img_A"]
        img_B = eval_df.iloc[i]["img_B"]
        gt    = eval_df.iloc[i]["label"]

        scores = []
        for img_path in [img_A, img_B]:
            image  = Image.open(img_path).convert("RGB")
            inputs = processor(
                text=prompt, images=image, return_tensors="pt"
            ).to("cuda", torch.float16)
            output = model.generate(
                **inputs, max_new_tokens=10, temperature=0.0, do_sample=False
            )
            score = parse_score(
                processor.decode(output[0], skip_special_tokens=True)
            )
            scores.append(score)

        score_A, score_B = scores

        THRESHOLD = 5  # Keep original threshold for livelier

        if score_A is None or score_B is None:
            final_pred, is_tie = None, False
        elif abs(score_A - score_B) <= THRESHOLD:
            final_pred, is_tie = None, True
        elif score_A > score_B:
            final_pred, is_tie = 1, False
        else:
            final_pred, is_tie = 0, False

        results.append({
            "pred":    final_pred,
            "gt":      gt,
            "tie":     is_tie,
            "score_A": score_A,
            "score_B": score_B
        })

    total    = sum(r["pred"] is not None for r in results)
    correct  = sum(r["pred"] == r["gt"] for r in results if r["pred"] is not None)
    acc      = correct / total if total > 0 else None
    tie_rate = sum(r["tie"] for r in results) / len(results) if results else 0

    scores_A = [r["score_A"] for r in results if r["score_A"] is not None]
    scores_B = [r["score_B"] for r in results if r["score_B"] is not None]
    if scores_A and scores_B:
        print(f"Score A — min: {min(scores_A)}, max: {max(scores_A)}, mean: {sum(scores_A)/len(scores_A):.1f}")
        print(f"Score B — min: {min(scores_B)}, max: {max(scores_B)}, mean: {sum(scores_B)/len(scores_B):.1f}")

    return {
        "attribute":            "livelier",
        "independent_acc":      round(acc, 3) if acc else None,
        "independent_tie_rate": round(tie_rate, 3)
    }

# Run it
res = evaluate_livelier_independent(attribute_map["livelier"], n_samples=100)
print("\n=== LIVELIER RESULTS ===")
print(res)

livelier — votes after study filter: 320803
livelier — votes after image filter: 318958
eval_df length: 100


Independent: livelier: 100%|██████████| 100/100 [00:41<00:00,  2.44it/s]

Score A — min: 0, max: 70, mean: 49.9
Score B — min: 0, max: 70, mean: 48.7

=== LIVELIER RESULTS ===
{'attribute': 'livelier', 'independent_acc': np.float64(0.714), 'independent_tie_rate': 0.93}


In [ ]:
cues     = ATTRIBUTE_CUES.get("livelier", [])
cue_text = ", ".join(cues)

prompt = f"""USER: <image>

Look at this street scene.

How lively does this place look?

Visual cues for liveliness: {cue_text}

Answer with ONLY one of these three options:
low
medium
high

A:"""

votes  = df_votes[df_votes["study_id"] == attribute_map["livelier"]].copy()
sample = votes[
    votes["left"].isin(image_map.keys()) &
    votes["right"].isin(image_map.keys())
].sample(5, random_state=42)

for _, row in sample.iterrows():
    for img_id, label in [(row["left"], "A"), (row["right"], "B")]:
        image  = Image.open(image_map[img_id]).convert("RGB")
        inputs = processor(
            text=prompt, images=image, return_tensors="pt"
        ).to("cuda", torch.float16)
        output = model.generate(
            **inputs, max_new_tokens=5, temperature=0.0, do_sample=False
        )
        raw   = processor.decode(output[0], skip_special_tokens=True)
        after = raw.split("A:")[-1].strip()
        print(f"Image {label} | Answer: {after}")
    print("---")

Image A | Answer: low
Image B | Answer: low
---
Image A | Answer: low
Image B | Answer: low
---
Image A | Answer: low
Image B | Answer: low
---
Image A | Answer: low
Image B | Answer: low
---
Image A | Answer: low
Image B | Answer: low
---


In [ ]:
# Debug: check LLaVA score distribution for more depressing
import re

cues     = ATTRIBUTE_CUES.get("more depressing", [])
cue_text = ", ".join(cues)

prompt = f"""USER: <image>

Rate how depressing this place looks on a scale from 0 to 100.

Important visual cues may include:
{cue_text}

Be precise and use the full range when appropriate.
Avoid default or middle values unless necessary.
Use a wide range of scores to clearly distinguish between different scenes.
Do not give identical scores unless the two places truly look the same.

Answer ONLY with a number between 0 and 100.

ASSISTANT:"""

votes  = df_votes[df_votes["study_id"] == attribute_map["more depressing"]].copy()
sample = votes[
    votes["left"].isin(image_map.keys()) &
    votes["right"].isin(image_map.keys())
].sample(10, random_state=42)

scores = []
for _, row in sample.iterrows():
    for img_id, label in [(row["left"], "A"), (row["right"], "B")]:
        image  = Image.open(image_map[img_id]).convert("RGB")
        inputs = processor(
            text=prompt, images=image, return_tensors="pt"
        ).to("cuda", torch.float16)
        output = model.generate(
            **inputs, max_new_tokens=10, temperature=0.0, do_sample=False
        )
        raw   = processor.decode(output[0], skip_special_tokens=True)
        after = raw.split("ASSISTANT:")[-1].strip()
        match = re.search(r"\d+", after)
        score = int(match.group()) if match else None
        scores.append(score)
        print(f"Image {label} | Score: {after}")
    print("---")

valid = [s for s in scores if s is not None]
if valid:
    print(f"\nScore distribution — min: {min(valid)}, max: {max(valid)}, mean: {sum(valid)/len(valid):.1f}")

Image A | Score: 50
Image B | Score: 50
---
Image A | Score: 50
Image B | Score: 50
---
Image A | Score: 70
Image B | Score: 80
---
Image A | Score: 70
Image B | Score: 50
---
Image A | Score: 80
Image B | Score: 50
---
Image A | Score: 50
Image B | Score: 50
---
Image A | Score: 50
Image B | Score: 50
---
Image A | Score: 50
Image B | Score: 50
---
Image A | Score: 70
Image B | Score: 70
---
Image A | Score: 70
Image B | Score: 70
---

Score distribution — min: 50, max: 80, mean: 59.0


In [ ]:
# Debug: see raw CoT output for beautiful
cues     = ATTRIBUTE_CUES.get("beautiful", [])
cue_text = ", ".join(cues)

reasoning_prompt = f"""USER: <image>

Two places are shown side-by-side:

Image A (left)
Image B (right)

Your task is to determine which place looks more **beautiful**.

Important visual cues related to "beautiful" may include:
{cue_text}

Follow these steps:

Step 1: Briefly describe Image A focusing on relevant cues.
Step 2: Briefly describe Image B focusing on relevant cues.
Step 3: Compare both images based on those cues.
Step 4: Decide which image better represents "beautiful".

IMPORTANT:
Give your final answer EXACTLY in this format on a new line:
Answer: A
or
Answer: B

ASSISTANT:
"""

votes  = df_votes[df_votes["study_id"] == attribute_map["beautiful"]].copy()
sample = votes[
    votes["left"].isin(image_map.keys()) &
    votes["right"].isin(image_map.keys())
].sample(5, random_state=42)

for idx, (_, row) in enumerate(sample.iterrows()):
    img_A = image_map[row["left"]]
    img_B = image_map[row["right"]]
    combined = concatenate_images(img_A, img_B)
    inputs = processor(
        text=reasoning_prompt, images=combined, return_tensors="pt"
    ).to("cuda", torch.float16)
    output = model.generate(
        **inputs, max_new_tokens=80, temperature=0.0, do_sample=False
    )
    raw = processor.decode(output[0], skip_special_tokens=True)
    after = raw.split("ASSISTANT:")[-1].strip()
    print(f"=== Sample {idx+1} ===")
    print(after)
    print()

=== Sample 1 ===
Answer: B

=== Sample 2 ===
Answer: A

=== Sample 3 ===
Answer: B

=== Sample 4 ===
Answer: A

=== Sample 5 ===
Answer: B

